### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex import Quinex
from quinex_application.visualisation_quinex import VisualisationQuinex
from quinex_application.oeo_data import OEOData
from quinex_application.levenshtein import LevenshteinDistance

In [ ]:
quinex = Quinex()
mongodb = MongoStore(quinex, database_name="quinex_db_query2", collection_name="papers_2025")
visualisation = VisualisationQuinex()
oeo = OEOData("../data/oeo/etd.xlsx", "../data/oeo/oeo-full.owl", "../data/oeo/oeo.xlsx")
levenshtein = LevenshteinDistance()

### **Load Analysis into DataFrame**

In [11]:
analysis = mongodb.get_collection("papers_2025")
df_path = "../data/quinex/df_quinex_2025.parquet"

if os.path.exists(df_path):
    df_quinex = visualisation.load_df(df_path)
else:
    df_quinex = visualisation.create_df(analysis)
    visualisation.save_df(df_quinex, df_path)

display(df_quinex.head())

📂 Loaded DataFrame from '../data/quinex/df_quinex_2025.parquet' with 64157 rows


,paper_id,entity,property,quantity,temporal_scope_text,spatial_scope_text
0,10.1038/s41597-025-04550-7,European high-voltage grid,voltage,> 200 kV,,
1,10.1038/s41597-025-04550-7,European high-voltage grid,energy system,= zero,,
2,10.1038/s41597-025-04550-7,our work,number of main advantages,= two main advantages,,
3,10.1038/s41597-025-04550-7,European,network development plan,Ten-Year,,
4,10.1038/s41597-025-04550-7,construction of the high-voltage grid,number of steps,= four steps,,


### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with OEO**

In [ ]:
df_oeo = oeo.process_oeo_data()
display(df_oeo.head())

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein_2025.parquet"

if os.path.exists(df_path):
    df_quinex = visualisation.load_df(df_path)
else:
    df_quinex[['best_oeo_match', 'match_score']] = df_quinex['entity'].apply(lambda x: pd.Series(levenshtein.get_best_match(x, df_oeo, "similarity")))
    visualisation.save_df(df_quinex, df_path)

display(df_quinex[(df_quinex['match_score'] >= 0.8) & (df_quinex['match_score'] < 1)].head(10))
print(len(df_quinex[(df_quinex['match_score'] >= 0.8) & (df_quinex['match_score'] < 1)]))

### **Visualise Results**

In [ ]:
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=25, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_values(df_quinex, column="best_oeo_match", top_n=25, ignore_empty=True, figsize=(6,6))

In [ ]:
fig = visualisation.plot_top_properties_of_entity(df_quinex, "electrolyzer", top_n=20, ignore_empty=True, figsize=(6,6))
fig, df = visualisation.plot_top_quantity_of_property(df_quinex, "electrolyzer", "rated power", top_n=8, ignore_empty=True, figsize=(6,6))

In [ ]:
display(df)

In [ ]:
value_counts = df["quantity"].value_counts().nlargest(10)
print(value_counts.values)
print(value_counts.index)